In [1]:
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, random_split
from torch.cuda.amp import GradScaler, autocast
import numpy as np
import zarr
import tqdm
import copy

# Optimized Loss-Functions

In [2]:
class DiffusionRegularizationLoss(nn.Module):
    # Penalizes sharp gradients in the displacement field to enforce smoothness.
    def __init__(self):
        super().__init__()

    def forward(self, dvf):
        # dvf shape: (B, C, D, H, W), where C=3 for vectors
        # Calculate gradients along each spatial dimension
        dy = torch.abs(dvf[:, :, 1:, :, :] - dvf[:, :, :-1, :, :])
        dx = torch.abs(dvf[:, :, :, 1:, :] - dvf[:, :, :, :-1, :])
        dz = torch.abs(dvf[:, :, :, :, 1:] - dvf[:, :, :, :, :-1])
        
        # Sum of squared gradients
        return (torch.mean(dx**2) + torch.mean(dy**2) + torch.mean(dz**2)) / 3.0

class CombinedLoss(nn.Module):
    # Combines a supervised loss (like MSE) with a regularization loss.
    def __init__(self, supervised_loss, regularization_loss, regularization_weight=0.01):
        super().__init__()
        self.supervised_loss = supervised_loss
        self.regularization_loss = regularization_loss
        self.lambda_ = regularization_weight

    def forward(self, predicted_dvf, ground_truth_dvf):
        supervised = self.supervised_loss(predicted_dvf, ground_truth_dvf)
        regularization = self.regularization_loss(predicted_dvf)
        total_loss = supervised + self.lambda_ * regularization
        return total_loss

## Model Architecture

In [3]:
class UNet3D(nn.Module):
    def __init__(self, in_channels=2, out_channels=3):
        super().__init__()
        self.enc1 = self._conv_block(in_channels, 16)
        self.enc2 = self._conv_block(16, 32)
        self.pool = nn.MaxPool3d(2)
        self.bottleneck = self._conv_block(32, 64)
        self.upconv2 = nn.ConvTranspose3d(64, 32, kernel_size=2, stride=2)
        self.dec2 = self._conv_block(64, 32)
        self.upconv1 = nn.ConvTranspose3d(32, 16, kernel_size=2, stride=2)
        self.dec1 = self._conv_block(32, 16)
        self.final_conv = nn.Conv3d(16, out_channels, kernel_size=1)
        
        # Initialize final layer to predict zero displacement
        self.final_conv.weight.data.zero_()
        self.final_conv.bias.data.zero_()

    def _conv_block(self, in_c, out_c):
        return nn.Sequential(
            nn.Conv3d(in_c, out_c, 3, 1, 1, bias=False),
            nn.InstanceNorm3d(out_c),
            nn.ReLU(inplace=True),
            nn.Conv3d(out_c, out_c, 3, 1, 1, bias=False),
            nn.InstanceNorm3d(out_c),
            nn.ReLU(inplace=True)
        )

    def forward(self, x_fixed, x_moving):
        x = torch.cat([x_fixed, x_moving], dim=1)
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool(e1))
        b = self.bottleneck(self.pool(e2))
        d2 = self.upconv2(b)
        d2 = torch.cat([d2, e2], dim=1)
        d2 = self.dec2(d2)
        d1 = self.upconv1(d2)
        d1 = torch.cat([d1, e1], dim=1)
        d1 = self.dec1(d1)
        return self.final_conv(d1)

# Data Loader

In [4]:
class SupervisedDisplacementDataset(Dataset):
    def __init__(self, moving_path, fixed_path, dvf_gt_path):
        super().__init__()
        self.moving_arr = zarr.open(moving_path, mode='r')
        self.fixed_arr = zarr.open(fixed_path, mode='r')
        self.dvf_gt_arr = zarr.open(dvf_gt_path, mode='r')
        
        # Check for consistent lengths
        self.num_images = self.moving_arr.shape[3]
        assert self.fixed_arr.shape[3] == self.num_images and self.dvf_gt_arr.shape[3] == self.num_images, \
            "The number of images and DVFs must match across all Zarr arrays."
            
        # Pre-calculate padding requirements
        self.original_shape = self.moving_arr.shape[:3] # H, W, D
        self.padded_shape = list(self.original_shape)
        for i in range(3):
            if self.padded_shape[i] % 4 != 0:
                self.padded_shape[i] = (self.padded_shape[i] // 4 + 1) * 4
    
    def __len__(self):
        return self.num_images

    def _pad_tensor(self, tensor):
        # Assumes tensor is (C, D, H, W)
        pad_d = self.padded_shape[2] - tensor.shape[1]
        pad_h = self.padded_shape[0] - tensor.shape[2]
        pad_w = self.padded_shape[1] - tensor.shape[3]
        padding = (pad_w // 2, pad_w - pad_w // 2, pad_h // 2, pad_h - pad_h // 2, pad_d // 2, pad_d - pad_d // 2)
        return F.pad(tensor, padding, "constant", 0)

    def __getitem__(self, idx):
        # Load data from Zarr arrays as numpy arrays
        moving_np = self.moving_arr[..., idx]
        fixed_np = self.fixed_arr[..., idx]
        dvf_gt_np = self.dvf_gt_arr[:, :, :, idx, :] # Shape (H, W, D, 3)

        # Convert to tensors
        moving_tensor = torch.from_numpy(moving_np.astype(np.float32)).permute(2, 0, 1).unsqueeze(0)
        fixed_tensor = torch.from_numpy(fixed_np.astype(np.float32)).permute(2, 0, 1).unsqueeze(0)
        dvf_gt_tensor = torch.from_numpy(dvf_gt_np.astype(np.float32)).permute(3, 2, 0, 1)

        # Apply padding
        moving_padded = self._pad_tensor(moving_tensor)
        fixed_padded = self._pad_tensor(fixed_tensor)
        dvf_gt_padded = self._pad_tensor(dvf_gt_tensor)

        return moving_padded, fixed_padded, dvf_gt_padded

## Training

In [5]:
def train_one_epoch(model, loader, optimizer, loss_fn, scaler, device, use_amp):
    model.train()
    total_loss = 0.0
    progress_bar = tqdm.tqdm(loader, desc="Training", leave=False)
    for moving, fixed, dvf_gt in progress_bar:
        moving, fixed, dvf_gt = moving.to(device), fixed.to(device), dvf_gt.to(device)
        optimizer.zero_grad(set_to_none=True)

        # KORREKTUR: autocast wird nur bei Bedarf aktiviert und ohne device_type
        with autocast(enabled=use_amp):
            predicted_dvf = model(fixed, moving)
            loss = loss_fn(predicted_dvf, dvf_gt)

        # Scaler-Operationen sind "no-ops" (tun nichts), wenn AMP deaktiviert ist
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        total_loss += loss.item()
        progress_bar.set_postfix(loss=f"{loss.item():.6f}")
    return total_loss / len(loader)

def validate(model, loader, loss_fn, device, use_amp):
    model.eval()
    total_loss = 0.0
    with torch.no_grad():
        for moving, fixed, dvf_gt in loader:
            moving, fixed, dvf_gt = moving.to(device), fixed.to(device), dvf_gt.to(device)
            
            # KORREKTUR: autocast wird auch hier verwendet
            with autocast(enabled=use_amp):
                predicted_dvf = model(fixed, moving)
                loss = loss_fn(predicted_dvf, dvf_gt)
            total_loss += loss.item()
    return total_loss / len(loader)

In [6]:
# --- Configuration ---
MODEL_LOAD_PATH = "best_supervised_model_250810_01_batch8_LR1e-4_Ep100_VSpl1_RegWeight01.pth"
MODEL_SAVE_PATH = "./best_supervised_model_250810_02_batch8_LR1e-5_Ep100_VSpl1_RegWeight01.pth"
MOVING_PATH = "../DCE_codeset/MRI-Datasets/DCE"
FIXED_PATH = "../DCE_codeset/MRI-Datasets/mdreg_DCE_fitting_results/coreg_zarr_2.zarr"
DVF_GT_PATH = "../DCE_codeset/MRI-Datasets/mdreg_DCE_fitting_results/transfo_zarr_2.zarr"

BATCH_SIZE = 8
LEARNING_RATE = 1e-5 # Slightly higher LR can be good with combined loss
NUM_EPOCHS = 100
VALIDATION_SPLIT = 0.1
REGULARIZATION_WEIGHT = 0.01 # This is the lambda (λ) factor

In [7]:
# --- Setup ---
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
full_dataset = SupervisedDisplacementDataset(MOVING_PATH, FIXED_PATH, DVF_GT_PATH)
test_size = int(VALIDATION_SPLIT * len(full_dataset))
train_size = len(full_dataset) - test_size
train_dataset, val_dataset = random_split(full_dataset, [train_size, test_size])

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=4, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=4, pin_memory=True)

model = UNet3D().to(device)
if os.path.exists(MODEL_LOAD_PATH):
    model.load_state_dict(torch.load(MODEL_LOAD_PATH, map_location=device))
    print(f"Loaded weights from {MODEL_LOAD_PATH}")
    
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)

# NEW: Use the combined loss function
loss_fn = CombinedLoss(
    supervised_loss=nn.MSELoss(),
    regularization_loss=DiffusionRegularizationLoss(),
    regularization_weight=REGULARIZATION_WEIGHT
).to(device)
# NEW: Learning Rate Scheduler
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, 'min', patience=5, factor=0.5)

use_amp = (device.type == 'cuda')
scaler = GradScaler(enabled=use_amp)
best_val_loss = float('inf')
best_model_weights = None

print(f"Starting supervised training on {device}...")
for epoch in range(NUM_EPOCHS):
    train_loss = train_one_epoch(model, train_loader, optimizer, loss_fn, scaler, device, use_amp)
    val_loss = validate(model, val_loader, loss_fn, device, use_amp)
    
    print(f"Epoch {epoch+1}/{NUM_EPOCHS} - Train Loss: {train_loss:.6f}, Val Loss: {val_loss:.6f}")
    
    # NEW: Learning rate scheduler step
    scheduler.step(val_loss)
    
    # NEW: Checkpointing
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_model_weights = copy.deepcopy(model.state_dict())
        torch.save(best_model_weights, MODEL_SAVE_PATH)
        print(f"  -> New best model saved to {MODEL_SAVE_PATH}")
print(f"Training complete. Best model saved to: {MODEL_SAVE_PATH} with validation loss {best_val_loss:.6f}")

Loaded weights from best_supervised_model_250810_01_batch8_LR1e-4_Ep100_VSpl1_RegWeight01.pth


/tmp/ipykernel_12725/2951726612.py:28: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler(enabled=use_amp)


Starting supervised training on cuda...


Training:   0%|          | 0/113 [00:00<?, ?it/s]/tmp/ipykernel_12725/503659223.py:10: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=use_amp):
/tmp/ipykernel_12725/503659223.py:31: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=use_amp):


Epoch 1/100 - Train Loss: 0.004671, Val Loss: 0.004534
  -> New best model saved to ./best_supervised_model_250810_02_batch8_LR1e-5_Ep100_VSpl1_RegWeight01.pth


...

Epoch 96/100 - Train Loss: nan, Val Loss: 0.005309


Epoch 97/100 - Train Loss: nan, Val Loss: 0.005309


Epoch 98/100 - Train Loss: 0.005407, Val Loss: 0.005309


Epoch 99/100 - Train Loss: 0.005407, Val Loss: nan


KeyboardInterrupt: 

In [8]:
test_indices = test_dataset.indices
indices_save_path = "test_indices2.pth"
torch.save(test_indices, indices_save_path)

NameError: name 'test_dataset' is not defined